# 01 — EDA: churn and the uplift experiment

> **Data sources:** Cell2Cell telecom churn (`jpacse/datasets-for-churn-telecom`, Kaggle mirror of the Teradata/Duke data) and the Hillstrom MineThatData e-mail experiment (minethatdata.com). **Downloaded:** 2026-08-10. Run `make data` to reproduce `data/raw/`.

Two datasets: Cell2Cell for churn/survival, Hillstrom for uplift. Logic is imported from `src/churn_uplift`.

In [1]:
from churn_uplift.data import CHURN_TARGET, load_cell2cell, load_hillstrom

churn = load_cell2cell()
print("Cell2Cell:", churn.shape, "| churn rate:", round(churn[CHURN_TARGET].mean(), 4))
churn[
    [
        "MonthsInService",
        "MonthlyRevenue",
        "MonthlyMinutes",
        "CustomerCareCalls",
        CHURN_TARGET,
    ]
].head()

Cell2Cell: (51047, 56) | churn rate: 0.2882


,MonthsInService,MonthlyRevenue,MonthlyMinutes,CustomerCareCalls,churn
0,61,24.00,219.0,0.0,1
1,58,16.99,10.0,0.0,1
2,60,38.00,8.0,0.0,0
3,59,82.28,1312.0,4.3,0
4,53,17.14,0.0,0.0,1


## Tenure distribution

Tenure (`MonthsInService`) is the duration used later for survival.

In [2]:
churn["MonthsInService"].describe().round(1)

count    51047.0
mean        18.8
std          9.8
min          6.0
25%         11.0
50%         16.0
75%         24.0
max         61.0
Name: MonthsInService, dtype: float64

## Churn rate by service-care intensity

Customers who call customer care more churn more — the first hint of the survival drivers.

In [3]:
import pandas as pd

care = pd.cut(
    churn["CustomerCareCalls"], [-1, 0, 1, 3, 999], labels=["0", "1", "2-3", "4+"]
)
churn.groupby(care, observed=True)[CHURN_TARGET].mean().round(3)

CustomerCareCalls
0      0.307
1      0.288
2-3    0.256
4+     0.250
Name: churn, dtype: float64

## The Hillstrom randomized experiment

Treatment (men's e-mail) vs. control is assigned at random, so the visit-rate gap is a causal effect.

In [4]:
h = load_hillstrom()  # Men's e-mail vs. control
h.groupby("treatment")["outcome"].agg(["count", "mean"]).round(4)

,count,mean
treatment,,
0,21306,0.1062
1,21307,0.1828
